## S&P 500 Investment Analysis

This notebook aims to calculate the total return of an initial $100 investment in the S&P 500 from January 1, 2005, to January 1, 2025. Additionally, it will identify the optimal continuous 11-calendar-day period to remove from this history to maximize returns.

In [1]:
# Install yfinance to fetch S&P 500 data
!pip install yfinance

In [2]:
import yfinance as yf
import pandas as pd
import numpy as np

# Define the date range and ticker
start_date = '2005-01-01'
end_date = '2025-01-01'
ticker_symbol = '^GSPC'  # S&P 500 ticker

# Fetch S&P 500 data
# We'll use 'Adj Close' to account for dividends and stock splits
sp500_data = yf.download(ticker_symbol, start=start_date, end=end_date)

# Display the first few rows of the data
display(sp500_data.head())

/tmp/ipykernel_355/2999225077.py:12: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500_data = yf.download(ticker_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2005-01-03,1202.079956,1217.800049,1200.319946,1211.920044,1510800000
2005-01-04,1188.050049,1205.839966,1185.390015,1202.079956,1721000000
2005-01-05,1183.739990,1192.729980,1183.719971,1188.050049,1738900000
2005-01-06,1187.890015,1191.630005,1183.270020,1183.739990,1569100000
2005-01-07,1186.189941,1192.199951,1182.160034,1187.890015,1477900000


In [9]:
# Calculate the total return for the initial $100 investment

# Ensure the index is a datetime object for proper sorting
sp500_data.index = pd.to_datetime(sp500_data.index)

# Sort the data by date to ensure correct calculation
sp500_data = sp500_data.sort_index()

# Get the 'Close' prices (for S&P 500, 'Close' is generally the relevant price)
adj_close_prices = sp500_data['Close']

# Get the price at the start and end of the investment period
initial_price = adj_close_prices.iloc[0]
final_price = adj_close_prices.iloc[-1]

# Calculate the total return
total_return = (final_price / initial_price) - 1

# Calculate the final value of the $100 investment
initial_investment = 100
final_investment_value = initial_investment * (1 + total_return)

# Extract scalar values for printing
initial_price_scalar = initial_price.item()
final_price_scalar = final_price.item()
total_return_scalar = total_return.item()
final_investment_value_scalar = final_investment_value.item()

print(f"Initial S&P 500 price on {adj_close_prices.index[0].strftime('%Y-%m-%d')}: ${initial_price_scalar:.2f}")
print(f"Final S&P 500 price on {adj_close_prices.index[-1].strftime('%Y-%m-%d')}: ${final_price_scalar:.2f}")
print(f"Total Return (including dividends): {total_return_scalar:.2%}")
print(f"Final value of initial ${initial_investment} investment: ${final_investment_value_scalar:.2f}")

Initial S&P 500 price on 2005-01-03: $1202.08
Final S&P 500 price on 2024-12-31: $5881.63
Total Return (including dividends): 389.29%
Final value of initial $100 investment: $489.29


## Optimal 11-Day Period Removal for Maximized Returns

Now, we will iterate through all possible continuous 11-calendar-day periods in the S&P 500 history and calculate the total return if that specific period were removed. The goal is to find the period whose removal results in the highest possible final investment value.

In [10]:
# Calculate returns by removing an 11-day period

def calculate_return_excluding_period(data, start_idx, end_idx, initial_investment):
    # Create a copy of the data and exclude the specified period
    data_excluded = pd.concat([data.iloc[:start_idx], data.iloc[end_idx:]])

    if len(data_excluded) < 2: # Not enough data points left for a meaningful return
        return -np.inf, None, None

    # Get the 'Close' prices from the filtered data
    prices_excluded = data_excluded['Close']

    # Calculate the return based on the remaining data
    initial_price_excluded = prices_excluded.iloc[0]
    final_price_excluded = prices_excluded.iloc[-1]

    total_return_excluded = (final_price_excluded / initial_price_excluded) - 1
    final_value_excluded = initial_investment * (1 + total_return_excluded)

    return final_value_excluded.item(), data_excluded.index[0], data_excluded.index[-1] # Return scalar

# Prepare data for iteration
data_points = sp500_data.shape[0]
window_size = 11  # 11 calendar days

max_final_value = final_investment_value.item() # Initialize with the original final value as scalar
best_removed_period_start = None
best_removed_period_end = None

# Convert index to a Series of dates for easier comparison and slicing
all_dates = pd.Series(sp500_data.index)

# Iterate through all possible 11-day windows to remove
for i in range(data_points):
    # Calculate the end index for the 11-day calendar period
    # We need to find the date that is 10 days after the current date (i.e., day 0 + 10 days = 11 days total)
    current_date = sp500_data.index[i]
    # This assumes we want to remove an 11-day *calendar* period, not necessarily 11 trading days
    end_date_of_window = current_date + pd.Timedelta(days=10)

    # Find the indices corresponding to these calendar dates in the trading data
    # We need to find all trading days between removal_start_date and removal_end_date (inclusive)
    indices_to_remove = sp500_data.index[
        (sp500_data.index >= current_date) & (sp500_data.index <= end_date_of_window)
    ]

    if indices_to_remove.empty:
        continue # No trading days in this 11-day calendar window

    first_idx_to_remove = sp500_data.index.get_loc(indices_to_remove.min())
    last_idx_to_remove = sp500_data.index.get_loc(indices_to_remove.max())

    # Calculate return after removing this period
    current_final_value, _, _ = calculate_return_excluding_period(
        sp500_data, first_idx_to_remove, last_idx_to_remove + 1, initial_investment
    )

    if current_final_value > max_final_value:
        max_final_value = current_final_value
        best_removed_period_start = current_date
        best_removed_period_end = end_date_of_window

# Calculate the return for the best removed period to get the percentage
if best_removed_period_start and best_removed_period_end:
    # Find the actual trading day indices corresponding to the best calendar period
    best_indices_to_remove = sp500_data.index[
        (sp500_data.index >= best_removed_period_start) & (sp500_data.index <= best_removed_period_end)
    ]
    first_best_idx = sp500_data.index.get_loc(best_indices_to_remove.min())
    last_best_idx = sp500_data.index.get_loc(best_indices_to_remove.max())

    final_val_best_removal, investment_start_date, investment_end_date = calculate_return_excluding_period(
        sp500_data, first_best_idx, last_best_idx + 1, initial_investment
    )
    best_return_percentage = (final_val_best_removal / initial_investment) - 1
else:
    final_val_best_removal = final_investment_value.item() # Ensure scalar
    best_return_percentage = total_return.item() # Ensure scalar


print(f"Original final value of initial ${initial_investment} investment: ${final_investment_value.item():.2f}")
print(f"Original total return: {total_return.item():.2%}")

if best_removed_period_start and best_removed_period_end:
    print(f"Best 11-calendar-day period to remove: {best_removed_period_start.strftime('%Y-%m-%d')} to {best_removed_period_end.strftime('%Y-%m-%d')}")
    print(f"New final value after removing this period: ${final_val_best_removal:.2f}")
    print(f"New total return after removing this period: {best_return_percentage:.2%}")
else:
    print("No 11-calendar-day period removal significantly improved returns, or no such period was found.")
    print(f"New final value: ${final_investment_value.item():.2f}")
    print(f"New total return: {total_return.item():.2%}")

# Save the results to a .txt file
output_filename = "investment_returns.txt"
with open(output_filename, "w") as f:
    f.write(f"Initial Investment: ${initial_investment}\n")
    f.write(f"Investment Period: {start_date} to {end_date}\n")
    f.write(f"\n")
    f.write(f"Original Investment Results:\n")
    f.write(f"  S&P 500 Price on {adj_close_prices.index[0].strftime('%Y-%m-%d')}: ${initial_price.item():.2f}\n")
    f.write(f"  S&P 500 Price on {adj_close_prices.index[-1].strftime('%Y-%m-%d')}: ${final_price.item():.2f}\n")
    f.write(f"  Total Return: {total_return.item():.2%}\n")
    f.write(f"  Final Value: ${final_investment_value.item():.2f}\n")
    f.write(f"\n")
    f.write(f"Results After Removing Optimal 11-Calendar-Day Period:\n")
    if best_removed_period_start and best_removed_period_end:
        f.write(f"  Period Removed: {best_removed_period_start.strftime('%Y-%m-%d')} to {best_removed_period_end.strftime('%Y-%m-%d')}\n")
        f.write(f"  New Total Return: {best_return_percentage:.2%}\n")
        f.write(f"  New Final Value: ${final_val_best_removal:.2f}\n")
    else:
        f.write(f"  No optimal 11-calendar-day period found to improve returns.\n")
        f.write(f"  New Total Return: {total_return.item():.2%}\n")
        f.write(f"  New Final Value: ${final_investment_value.item():.2f}\n")

print(f"Results saved to {output_filename}")

Original final value of initial $100 investment: $489.29
Original total return: 389.29%
Best 11-calendar-day period to remove: 2024-12-26 to 2025-01-05
New final value after removing this period: $502.47
New total return after removing this period: 402.47%
Results saved to investment_returns.txt


In [11]:
# Display the content of the output file
with open("investment_returns.txt", "r") as f:
    print(f.read())

Initial Investment: $100
Investment Period: 2005-01-01 to 2025-01-01

Original Investment Results:
  S&P 500 Price on 2005-01-03: $1202.08
  S&P 500 Price on 2024-12-31: $5881.63
  Total Return: 389.29%
  Final Value: $489.29

Results After Removing Optimal 11-Calendar-Day Period:
  Period Removed: 2024-12-26 to 2025-01-05
  New Total Return: 402.47%
  New Final Value: $502.47

